# 03 - WoE and IV for Win/Loss

This notebook runs variable-wise optimal binning for `Opportunity Result` (Won/Loss), computes WoE bins, and reports Information Value (IV) for all candidate predictors.

In [6]:
import pandas as pd
import numpy as np
from pathlib import Path
from optbinning import OptimalBinning

pd.set_option('display.max_columns', 200)

In [7]:
df_load = pd.read_parquet('../../data/intermediate/df_train_stratified.parquet')
print('shape:', df_load.shape)
df_load.head(2)

shape: (54617, 37)


,Opportunity Number,Supplies Group,Supplies Subgroup,Region,Route To Market,Elapsed Days In Sales Stage,Opportunity Result,Sales Stage Change Count,Total Days Identified Through Closing,Total Days Identified Through Qualified,Opportunity Amount USD,Client Size By Revenue (USD),Client Size By Employee Count,Revenue From Client Past Two Years (USD),Competitor Type,Ratio Days Identified To Total Days,Ratio Days Validated To Total Days,Ratio Days Qualified To Total Days,Deal Size Category (USD),total_days_zero,Opportunity Result Bool,opportunity_amount_weirdness,row_position,flag_0_days,flag_ratio_problem,flag_zero_opportunity_amount,flag_outlier_opportunity_amount,flag_outlier_total_days,flag_totally_repeated_row,flag_partially_repeated_row,partial_repeat_is_latest_id_appearance,flag_only_identified,flag_weirdness_over_75th_pct,problem_tags,problem_count,amount_qbin,stratify_key
0,7062187,Car Accessories,Towing & Hitches,Northeast,Fields Sales,41,Loss,4,41,41,200000,100K or less,1K or less,"25,000 - 50,000",NaN,0.84058,0.15942,0.0,40K to 50K,False,False,19.931465,55880,False,False,False,False,False,False,False,False,False,True,[weirdness_over_75th_pct],1,"(131000.0, 220000.0]","0__(131000.0, 220000.0]"
1,9718029,Car Accessories,Garage & Car Care,Midwest,Telesales,7,Loss,2,6,6,28763,100K or less,1K or less,0 (No business),Unknown,1.00000,0.00000,0.0,20K to 30K,False,False,4.578548,43421,False,False,False,False,False,False,False,False,True,False,[flag_only_identified],1,"(20000.0, 30000.0]","0__(20000.0, 30000.0]"


In [8]:
columns_to_use = ['Supplies Group', 'Supplies Subgroup', 'Region',
       'Route To Market', 'Elapsed Days In Sales Stage', 'Opportunity Result',
       'Sales Stage Change Count', 'Total Days Identified Through Closing',
       'Total Days Identified Through Qualified', 'Opportunity Amount USD',
       'Client Size By Revenue (USD)', 'Client Size By Employee Count',
       'Revenue From Client Past Two Years (USD)', 'Competitor Type',
       'Ratio Days Identified To Total Days',
       'Ratio Days Validated To Total Days',
       'Ratio Days Qualified To Total Days', 'Deal Size Category (USD)', "Opportunity Result Bool"]

df = df_load[columns_to_use].copy()

In [9]:

df['target_win'] = df['Opportunity Result'].map({'Won': 1, 'Loss': 0})
print('shape:', df.shape)
print('target nulls:', df['target_win'].isna().sum())

shape: (54617, 20)
target nulls: 0


## Fit WoE/IV by variable

In [10]:
excluded = {'Opportunity Result', 'target_win', 'Opportunity Number'}
features = [c for c in df.columns if c not in excluded]

results = []
failed = []
all_variable_tables = []
for col in features:
    x = df[col]
    y = df['target_win']
    dtype = 'numerical' if pd.api.types.is_numeric_dtype(x) else 'categorical'

    try:
        optb = OptimalBinning(name=col, dtype=dtype)
        optb.fit(x, y)
        bt = optb.binning_table.build()
        iv_total = float(bt.iloc[-1]['IV']) if 'IV' in bt.columns else np.nan
        js_total = float(bt.iloc[-1]['JS']) if 'JS' in bt.columns else np.nan
        bt_copy = bt.copy()
        bt_copy.insert(0, 'variable', col)
        bt_copy.insert(1, 'dtype', dtype)
        bt_copy.insert(2, 'status', optb.status)
        all_variable_tables.append(bt_copy)

        results.append({
            'variable': col,
            'dtype': dtype,
            'status': optb.status,
            'iv': iv_total,
            'js': js_total
        })
    except Exception as e:
        failed.append({'variable': col, 'error': str(e)[:200]})

iv_df = pd.DataFrame(results)
if not iv_df.empty:
    iv_df = iv_df.sort_values('iv', ascending=False).reset_index(drop=True)
failed_df = pd.DataFrame(failed)
all_variable_info_df = pd.concat(all_variable_tables, ignore_index=True) if all_variable_tables else pd.DataFrame()
print('processed:', len(iv_df), 'failed:', len(failed_df))
print('all variable rows:', len(all_variable_info_df))
iv_df.head(20)

processed: 18 failed: 0
all variable rows: 145


,variable,dtype,status,iv,js
0,Total Days Identified Through Qualified,numerical,OPTIMAL,0.806683,0.095527
1,Total Days Identified Through Closing,numerical,OPTIMAL,0.696991,0.083224
2,Revenue From Client Past Two Years (USD),categorical,OPTIMAL,0.578196,0.065006
3,Ratio Days Identified To Total Days,numerical,OPTIMAL,0.391933,0.044279
4,Opportunity Amount USD,numerical,OPTIMAL,0.333148,0.039772
5,Ratio Days Qualified To Total Days,numerical,OPTIMAL,0.315775,0.038751
6,Deal Size Category (USD),categorical,OPTIMAL,0.273831,0.033433
7,Ratio Days Validated To Total Days,numerical,OPTIMAL,0.233745,0.028613
8,Sales Stage Change Count,numerical,OPTIMAL,0.117576,0.014573
9,Supplies Subgroup,categorical,OPTIMAL,0.065448,0.008129


## IV interpretation bands

In [11]:
def iv_band(iv):
    if iv < 0.02:
        return 'Not useful'
    if iv < 0.1:
        return 'Weak'
    if iv < 0.3:
        return 'Medium'
    if iv < 0.5:
        return 'Strong'
    return 'Suspicious/Very strong'

iv_df['iv_band'] = iv_df['iv'].apply(iv_band)
iv_df

,variable,dtype,status,iv,js,iv_band
0,Total Days Identified Through Qualified,numerical,OPTIMAL,0.806683,0.095527,Suspicious/Very strong
1,Total Days Identified Through Closing,numerical,OPTIMAL,0.696991,0.083224,Suspicious/Very strong
2,Revenue From Client Past Two Years (USD),categorical,OPTIMAL,0.578196,0.065006,Suspicious/Very strong
3,Ratio Days Identified To Total Days,numerical,OPTIMAL,0.391933,0.044279,Strong
4,Opportunity Amount USD,numerical,OPTIMAL,0.333148,0.039772,Strong
5,Ratio Days Qualified To Total Days,numerical,OPTIMAL,0.315775,0.038751,Strong
6,Deal Size Category (USD),categorical,OPTIMAL,0.273831,0.033433,Medium
7,Ratio Days Validated To Total Days,numerical,OPTIMAL,0.233745,0.028613,Medium
8,Sales Stage Change Count,numerical,OPTIMAL,0.117576,0.014573,Medium
9,Supplies Subgroup,categorical,OPTIMAL,0.065448,0.008129,Weak


In [12]:
output_dir = Path('../../data/bivariate_info')
excel_path = output_dir / 'win_loss_iv_report.xlsx'

with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    iv_df.to_excel(writer, sheet_name='variable_summaries', index=False)
    all_variable_info_df.to_excel(writer, sheet_name='all_variable_info', index=False)

print('saved excel:', excel_path)
print('summary rows:', len(iv_df))
print('all-variable rows:', len(all_variable_info_df))
print('saved failures:', (not failed_df.empty))

saved excel: ../../data/bivariate_info/win_loss_iv_report.xlsx
summary rows: 18
all-variable rows: 145
saved failures: False


## Top variable WoE table example

In [13]:
if iv_df.empty:
    print('No successful WoE/IV fits to display.')
else:
    top_var = iv_df.loc[0, 'variable']
    x = df[top_var]
    y = df['target_win']
    dtype = 'numerical' if pd.api.types.is_numeric_dtype(x) else 'categorical'
    optb = OptimalBinning(name=top_var, dtype=dtype)
    optb.fit(x, y)
    woe_table = optb.binning_table.build()
    print('Top variable:', top_var)
    display(woe_table)

Top variable: Total Days Identified Through Qualified


,Bin,Count,Count (%),Non-event,Event,Event rate,WoE,IV,JS
0,"(-inf, 0.50)",6689,0.122471,3630,3059,0.457318,-1.060357,0.171834,0.020526
1,"[0.50, 3.50)",6531,0.119578,3601,2930,0.448630,-1.025293,0.156136,0.018705
2,"[3.50, 6.50)",5541,0.101452,3679,1862,0.336040,-0.550512,0.035169,0.004341
3,"[6.50, 8.50)",3276,0.059981,2424,852,0.260073,-0.185914,0.002178,0.000272
4,"[8.50, 10.50)",3137,0.057436,2494,643,0.204973,0.123997,0.000853,0.000107
5,"[10.50, 12.50)",2785,0.050991,2235,550,0.197487,0.170576,0.001414,0.000177
6,"[12.50, 15.50)",3934,0.072029,3411,523,0.132944,0.643677,0.024649,0.003029
7,"[15.50, 18.50)",3499,0.064064,3054,445,0.127179,0.694631,0.025126,0.003079
8,"[18.50, 22.50)",4304,0.078803,3907,397,0.092240,1.055087,0.063556,0.007595
9,"[22.50, 26.50)",3490,0.063900,3181,309,0.088539,1.100108,0.055223,0.006575
